# Week 21, Day 5: GitHub and pull requests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w021d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
mine = ["c1", "c2", "y3", "y4"]
github = ["c1", "c2", "t3"]

ahead = [c for c in mine if c not in github]
behind = [c for c in github if c not in mine]
print("ahead:", ahead)
print("behind:", behind)

> **Common mistake.** A token or key is as powerful as your password. Never paste one into code, a commit, a chat or a public place. If one is ever exposed, delete it on GitHub at once and make a new one. The same goes for any API key: keep it in an environment variable or a `.env` file that's listed in `.gitignore`.

*Interactive figure on the website: Your branch and its copy on GitHub. Try pushing when GitHub has a commit you don't..*

> **Key idea.** A remote is another copy of the repository, usually `origin` on GitHub. `git push` sends your commits, `git pull` fetches and merges theirs; a push is rejected if the remote has work you don't, so pull first. Teams change `main` through pull requests: push a branch, open a PR, get it reviewed, merge. Keep tokens and keys out of every repository.

### Exercise: Who and what

Write `owner_and_repo(url)` returning `(owner, repo)` from a GitHub URL in either form: `https://github.com/OWNER/REPO.git`, `https://github.com/OWNER/REPO` (no `.git`), or `git@github.com:OWNER/REPO.git`. Return `None` for anything that isn't a GitHub repository URL.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert owner_and_repo(\"https://github.com/peter-adepoju/python-atlas.git\") == (\"peter-adepoju\", \"python-atlas\")\n    assert owner_and_repo(\"https://github.com/python/cpython\") == (\"python\", \"cpython\")\n    assert owner_and_repo(\"git@github.com:python/cpython.git\") == (\"python\", \"cpython\")\n    assert owner_and_repo(\"https://gitlab.com/a/b.git\") is None and owner_and_repo(\"https://github.com/onlyowner\") is None")

### Exercise: Ahead and behind

Write `ahead_behind(local, remote)` where each argument is a list of commit IDs from oldest to newest, and return `(ahead, behind)`: how many local commits the remote doesn't have, and how many remote commits the local doesn't have. Then `advice(local, remote)` returning one of `"up to date"`, `"push"`, `"pull"` or `"pull, then push"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert ahead_behind([\"c1\", \"c2\", \"y3\", \"y4\"], [\"c1\", \"c2\", \"t3\"]) == (2, 1)\n    assert advice([\"c1\", \"c2\", \"y3\", \"y4\"], [\"c1\", \"c2\", \"t3\"]) == \"pull, then push\"\n    assert advice([\"c1\"], [\"c1\"]) == \"up to date\" and advice([\"c1\", \"y2\"], [\"c1\"]) == \"push\" and advice([\"c1\"], [\"c1\", \"t2\"]) == \"pull\"")

### Exercise: Rejected!

Model a remote as a list of commit IDs. Write `push(local, remote)` that returns the remote's new list: if every remote commit is already in `local`, the remote becomes a copy of `local`; otherwise raise `ValueError("rejected: pull first")`. Then `pull(local, remote, merge_id)`: if the local has nothing the remote lacks, return a copy of `remote` (a fast-forward); if the remote has nothing new, return `local` unchanged; otherwise return the remote's commits, then the local-only commits, then `merge_id`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert push([\"c1\", \"y2\"], [\"c1\"]) == [\"c1\", \"y2\"]\n    try:\n        push([\"c1\", \"y2\"], [\"c1\", \"t2\"])\n        assert False, \"should be rejected\"\n    except ValueError as e:\n        assert \"pull first\" in str(e)\n    assert pull([\"c1\"], [\"c1\", \"t2\"], \"m\") == [\"c1\", \"t2\"]\n    assert pull([\"c1\", \"y2\"], [\"c1\"], \"m\") == [\"c1\", \"y2\"]\n    merged = pull([\"c1\", \"y2\"], [\"c1\", \"t2\"], \"m3\")\n    assert merged == [\"c1\", \"t2\", \"y2\", \"m3\"] and push(merged, [\"c1\", \"t2\"]) == merged")

### Exercise: A pull request checklist

Write `pr_problems(title, description, changed_lines, branch)` returning a list of problems with a pull request (an empty list if it's fine), in this order: `"title too long"` if the title is over 72 characters; `"title is vague"` if, ignoring case, it's one of `"update"`, `"fix"`, `"changes"`, `"wip"`; `"no description"` if the description is empty or only spaces; `"too big to review"` if more than 400 lines changed; `"merging from main"` if the branch is `"main"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert pr_problems(\"Fix crash on empty marks file\", \"Return 0 when there are no marks.\", 12, \"fix-empty\") == []\n    assert pr_problems(\"fix\", \" \", 900, \"main\") == [\"title is vague\", \"no description\", \"too big to review\", \"merging from main\"]\n    assert pr_problems(\"x\" * 73, \"ok\", 1, \"b\") == [\"title too long\"]")

### Exercise: Catch a secret before it's pushed

Before pushing, some teams scan changes for secrets. Write `find_secrets(files)` taking a dictionary from file name to contents and returning a sorted list of `(file, line_number)` pairs (lines counted from 1) where a line contains `"api_key"`, `"password"` or `"token"` (ignoring case) followed somewhere by `=` and then a quoted string of at least 8 characters, using the regular expression in the starter. Files named `.env` are skipped, since they're meant to hold secrets (and must be in `.gitignore`).

In [ ]:
import re

PATTERN = re.compile(r"(api_key|password|token)\s*=\s*['\"][^'\"]{8,}['\"]", re.IGNORECASE)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    files = {\n        \"app.py\": \"import os\\nAPI_KEY = 'sk-123456789abc'\\nkey = os.environ['API_KEY']\\n\",\n        \"db.py\": \"password = \\\"short\\\"\\nPassword='longer-than-8'\\n\",\n        \".env\": \"TOKEN='abcdefghijklmnop'\\n\",\n        \"README.md\": \"Set your token = in the environment.\\n\",\n    }\n    assert find_secrets(files) == [(\"app.py\", 2), (\"db.py\", 2)]\n    assert find_secrets({}) == []")

### Bug hunt: Overwriting the team's work

This model of `push` lets a push through even when the remote has a commit you don't, which would throw away the teammate's work. It should print `rejected`. Fix the check.

In [ ]:
def push(local, remote):
    if len(local) >= len(remote):
        return list(local)
    raise ValueError("rejected")

try:
    print(push(["c1", "y2"], ["c1", "t2"]))
except ValueError as e:
    print(e)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"rejected\"\n    expect_output(\"rejected\")\n    assert push([\"c1\", \"y2\"], [\"c1\"]) == [\"c1\", \"y2\"]")

### Bug hunt: A key in the code

This script has an API key written into it, so it would be published with the code. Change it to read the key from the environment variable `WEATHER_KEY` with `os.environ.get`, using `"missing"` when it isn't set, so it prints `key set: False` here (where the variable isn't set). The key's text must not appear anywhere in the code.

In [ ]:
import os

API_KEY = "wk-4f9a77c2d8e14b2a"
print("key set:", API_KEY != "missing")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"from the environment\"\n    expect_output(\"key set: False\")\n    assert \"wk-4f9a77c2d8e14b2a\" not in source and \"WEATHER_KEY\" in source")

### Puzzle: Contributing a fix

Arrange the commands to contribute a fix through a pull request. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # the usual name of the main remote
b = ___   # the command that downloads new commits without changing your files (two words)
c = ___   # True or False: git refuses a push when the remote has commits you don't have
d = ___   # the file GitHub shows on a repository's front page (its name)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"origin\" and b == \"git fetch\" and c is True and d == \"README.md\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a review helper

GitHub shows changes as a **unified diff**. Write a helper that reads one and finds things worth a reviewer's attention. Write `changed_lines(diff_text)` returning a list of `(file, line_number, text)` for every **added** line, where `file` comes from the `+++ b/NAME` line and the line numbers come from the hunk headers `@@ -a,b +c,d @@` (the new file's lines start at `c`; added lines `+` and unchanged lines (starting with a space) count up the new file's line number; removed lines `-` don't). Then `review(diff_text)` returning a list of comments `"file:line: message"` for added lines that contain `print(` (`"debug print left in?"`), `TODO` (`"unfinished TODO"`) or are longer than 79 characters (`"line too long"`), in that order of checks, one comment per problem.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("DIFF = \"\"\"diff --git a/stats.py b/stats.py\n--- a/stats.py\n+++ b/stats.py\n@@ -1,4 +1,6 @@\n def mean(xs):\n-    return sum(xs) / len(xs)\n+    if not xs:\n+        return 0\n+    print(\"debug\", xs)\n+    return sum(xs) / len(xs)\n \n def spread(xs):\n@@ -10,2 +12,3 @@\n # end\n+# TODO: add median\ndiff --git a/README.md b/README.md\n--- a/README.md\n+++ b/README.md\n@@ -1,1 +1,2 @@\n # Stats\n+\"\"\" + \"x\" * 85 + \"\\n\"\ndef test_lines():\n    \"changed_lines\"\n    lines = changed_lines(DIFF)\n    assert lines[0] == (\"stats.py\", 2, \"    if not xs:\")\n    assert (\"stats.py\", 4, '    print(\"debug\", xs)') in lines\n    assert (\"stats.py\", 13, \"# TODO: add median\") in lines\n    assert (\"README.md\", 2, \"x\" * 85) in lines and len(lines) == 6\ndef test_review():\n    \"review\"\n    assert review(DIFF) == [\"stats.py:4: debug print left in?\", \"stats.py:13: unfinished TODO\", \"README.md:2: line too long\"]")

## Warm-up

You have two commits GitHub doesn't, and GitHub has one you don't. Git calls that **ahead by 2, behind by 1**, and that one line in `git status` tells you exactly what to do: get theirs, then send yours.

## Remotes

A **remote** is another copy of the same repository, somewhere else: usually on GitHub (or GitLab, or Bitbucket). The usual name for the main remote is **`origin`**.

- `git clone URL` copies a repository from GitHub to your computer, with all its history, and sets up `origin` automatically.
- `git remote -v` lists the remotes and their URLs.
- `git remote add origin URL` connects an existing local repository to a new, empty GitHub one.

Repository URLs come in two forms, and both point to the same repository:

```
https://github.com/peter-adepoju/python-atlas.git
git@github.com:peter-adepoju/python-atlas.git
```

The first uses HTTPS (you sign in with a **personal access token**, not your password); the second uses SSH (you add an SSH key to your GitHub account once). Either is fine.

## Push and pull

| Command | Does |
|---|---|
| `git push` | sends your new commits on this branch to the remote |
| `git push -u origin name` | the first push of a new branch, linking it to the remote (later, just `git push`) |
| `git fetch` | downloads new commits from the remote, without changing your files |
| `git pull` | `git fetch`, then merge the remote branch into yours |

Your computer and GitHub are two copies that only meet when you push or pull. Try it here: commit, push, have a teammate push, and see what happens when you both have new work:

The key rule: **if the remote has commits you don't, Git refuses your push** ("rejected, fetch first"), so that nobody's work is silently overwritten. Pull first, which merges their work with yours (fix any conflicts as you learned yesterday), then push.

A good habit when working with others: `git pull` before you start work each day, and push when you finish.

## Pull requests

A **pull request** (PR) is a request to merge one branch into another, with a page on GitHub to discuss it. It's how almost every team makes changes:

1. `git switch -c fix-empty-file` and make your change in commits.
2. `git push -u origin fix-empty-file`.
3. On GitHub, open a pull request from your branch into `main`, with a title and a description: **what** changed, **why**, and how you checked it.
4. Teammates **review** it: they read the changes (GitHub shows the diff), comment on lines, and approve or ask for changes. Automated checks (tests) often run too.
5. You push more commits to the same branch to answer the review; the pull request updates by itself.
6. When approved, it's merged on GitHub, and the branch is deleted.
7. Everyone does `git pull` on `main` to get it.

Pull requests catch bugs before they reach `main`, spread knowledge around the team, and leave a written record of why each change was made.

**Forks.** To suggest a change to a project you can't push to (most open-source projects), you **fork** it: GitHub makes your own copy under your account. You push a branch to your fork and open a pull request from it to the original. That's how you'll contribute to open source later in this course.

**Issues** are GitHub's to-do lists and bug reports, one page per problem or idea. A pull request can close an issue automatically by writing "Fixes #42" in its description.

**README and licence.** Every repository should have a `README.md` at the top level: what the project is, how to install and run it, and how to contribute. GitHub shows it on the repository's front page. A **licence** file says what others may do with your code; without one, others legally can't reuse it. The MIT licence is a short, common choice for open source.

## Reviewing code

Reviewing is a skill. Good review comments are:

- **specific:** point at the line, and say what and why: "This divides by `len(marks)`, which crashes for an empty list. Could we return 0 there?"
- **kind and about the code**, not the person: "this function" rather than "you".
- **sorted by importance:** bugs and unclear design first; style last (and leave style to automatic tools where possible, Week 22).
- **questions where you're unsure:** "Is there a reason for copying the list here?" Often there is.

And as the author: keep pull requests **small** (a few hundred lines at most; a reviewer can't check 3,000 lines properly), describe them well, and treat review comments as help, not criticism.

## On your own computer

1. Create a free account at github.com.
2. On GitHub, click **New repository**, name it `atlas`, leave it **empty** (no README), and create it.
3. In your terminal, in your `atlas` folder (which already has commits from Day 3):
   ```
   git remote add origin https://github.com/YOUR-NAME/atlas.git
   git push -u origin main
   ```
   When asked to sign in, follow GitHub's instructions to use a browser sign-in or a personal access token (Settings, Developer settings, Personal access tokens). Never type the token anywhere but the sign-in prompt.
4. Refresh the GitHub page: your files and history are there.
5. Make a branch, commit a small change to a `README.md`, push the branch with `git push -u origin your-branch`, and open a pull request to `main` on GitHub. Review your own pull request, then merge it, and `git pull` on `main` locally.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at this course's own repository, peter-adepoju/python-atlas on GitHub: open the commit history, click into a commit to see its diff, and look at how the files are organised. Every page you've studied on is there as plain text. Tomorrow is the Forge: you'll build a small version-control tool that brings the whole week together.